# Rfam Tools

![Rfam](https://proto-bio.github.io/proto-assets/images/tool/rfam/hero.png)

This notebook demonstrates the two Rfam wrappers provided by proto_tools. `run_rfam_regions` lists where an RNA family is annotated in sequenced genomes, with coordinates and strand, and `run_rfam_family` fetches a family's curation record and consensus secondary structure. The running example is TwoAYGGAY (RF01731), a bacterial regulatory RNA whose two hairpins carry the AYGGAY motif, in *Pseudomonas fluorescens* SBW25.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("rfam")
display_overview("rfam")
display_docs_section("rfam", "Background")

## Available tools

In [ ]:
display_available_tools("rfam")

### `run_rfam_regions`

`run_rfam_regions` lists every sequence region a family is annotated in. Each hit carries its sequence accession, bit score, 1-indexed inclusive coordinates with `start <= end`, strand, species, and NCBI taxonomy ID. Filters narrow the list to one taxonomy ID, a species substring, or a sequence accession. Here we list the TwoAYGGAY hits in *P. fluorescens* SBW25 (taxonomy ID 216595).

In [ ]:
from proto_tools import (
    RfamRegionsConfig,
    RfamRegionsInput,
    run_rfam_regions,
)

In [ ]:
# Display docs
display_api_reference("rfam", "input", "run_rfam_regions")

# TwoAYGGAY hits in Pseudomonas fluorescens SBW25
regions_inputs = RfamRegionsInput(family="RF01731", taxid=216595)

In [ ]:
# Display config docs
display_api_reference("rfam", "config", "run_rfam_regions")

# Defaults return up to 500 regions | see docs above for all fields
regions_config = RfamRegionsConfig()

In [ ]:
# Run the lookup
regions_result = run_rfam_regions(regions_inputs, regions_config)

In [ ]:
# Display output docs
display_api_reference("rfam", "output", "run_rfam_regions")

print(f"{regions_result.rfam_id} ({regions_result.accession}), Rfam {regions_result.rfam_release}")
print(f"{regions_result.matched_regions} of {regions_result.total_regions} regions are in SBW25")
for region in sorted(regions_result.regions, key=lambda r: -r.bit_score)[:5]:
    print(f"  {region.sequence_accession} {region.start}-{region.end} ({region.strand})  {region.bit_score} bits")

#### Pull a Locus with Flanks

The coordinates line up with `run_ncbi_efetch`'s `seq_start`, `seq_stop`, and `strand`, so the best-scoring hit can be fetched directly, here with 200 nt of flanking sequence on each side.

In [ ]:
from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

top = max(regions_result.regions, key=lambda r: r.bit_score)
flank = 200
locus = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore",
        identifier=top.sequence_accession,
        seq_start=top.start - flank,
        seq_stop=top.end + flank,
        strand=top.strand,
    ),
    NCBIEfetchConfig(),
)
record = locus.fasta_records[0]
print(record.header)
print(f"Length: {len(record.sequence)} nt (hit {top.end - top.start + 1} nt + {2 * flank} nt flanks)")

### `run_rfam_family`

`run_rfam_family` fetches a family by accession or ID. It returns the curation record (type, authors, seed and structure sources, counts, clan, and bit-score cutoffs) and the consensus secondary structure with its reference sequence, read from the seed alignment. The full Stockholm seed alignment is returned when `include_seed_alignment` is set.

In [ ]:
from proto_tools import (
    RfamFamilyConfig,
    RfamFamilyInput,
    run_rfam_family,
)

In [ ]:
# Display docs
display_api_reference("rfam", "input", "run_rfam_family")

# The family ID works as well as the accession
family_inputs = RfamFamilyInput(family="TwoAYGGAY")

In [ ]:
# Display config docs
display_api_reference("rfam", "config", "run_rfam_family")

# Also return the full seed alignment | see docs above for all fields
family_config = RfamFamilyConfig(include_seed_alignment=True)

In [ ]:
# Run the fetch
family_result = run_rfam_family(family_inputs, family_config)

In [ ]:
# Display output docs
display_api_reference("rfam", "output", "run_rfam_family")

print(f"{family_result.accession} {family_result.rfam_id}: {family_result.description} ({family_result.rna_type})")
print(f"Seed {family_result.num_seed}, full {family_result.num_full}, species {family_result.num_species}, clan {family_result.clan_id}")
print(f"Gathering cutoff: {family_result.gathering_cutoff} bits")
print()
print(family_result.consensus_structure)
print(family_result.consensus_sequence)

## Export Results

Regions export to JSON, TSV, or CSV, and the family exports to JSON or, when the seed alignment was requested, to Stockholm (`sto`).

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

regions_result.export("rfam_regions_sbw25", export_path=output_dir, file_format="tsv")
print(f"Exported to {output_dir / 'rfam_regions_sbw25.tsv'}")

family_result.export("rfam_family_rf01731", export_path=output_dir, file_format="sto")
print(f"Exported to {output_dir / 'rfam_family_rf01731.sto'}")